# [Cell 1] Markdown: 說明與架構

蘇東坡 AI 雙模組協作中轉站（純本地 Multi-Agent + 快取優化版）

**架構**：Qwen2-7B-Instruct (審核者) → 判斷時間線 → 動態生成 Prompt → 本地東坡 LoRA (扮演者)。

- 完全移除對外 API 依賴，全程在 Colab T4 運行。
- 雙 7B 模型皆採用 4-bit 量化載入，總 VRAM 佔用約 10GB（T4 極限為 15GB）。
- **【新增】模型快取機制**：自動檢查雲端硬碟是否有 Qwen 模型，並在載入前先複製到 Colab 本地空間，大幅提升載入速度並減少雲端硬碟 I/O 負擔。

## [Cell 2] Code: 安裝依賴套件

In [ ]:
# 移除 google-genai，保留本地端推論所需套件。
# 增加 huggingface_hub 用於模型下載
%pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub
print("套件安裝完成")

## [Cell 3] Code: 掛載 Drive 與設定路徑

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# === 你的東坡模型路徑 ===
BASE_MODEL_PATH = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Base"
LORA_WEIGHT_PATH = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/DongPo-Adapter"

# === 審核者模型 (Qwen2-7B-Instruct) 雲端硬碟與本地快取路徑 ===
DRIVE_OBSERVER_PATH = "/content/drive/MyDrive/FJU_dongpoProject/dongPo/models/observer/Qwen2-7B-Instruct"
LOCAL_OBSERVER_PATH = "/content/local_models/Qwen2-7B-Instruct"

assert os.path.isdir(BASE_MODEL_PATH), f"找不到東坡底模資料夾：{BASE_MODEL_PATH}"
assert os.path.isdir(LORA_WEIGHT_PATH), f"找不到東坡 adapter 資料夾：{LORA_WEIGHT_PATH}"
print("基本路徑設定完成。")

## [Cell 4] Code: 自動檢查下載與建立本地快取 (優化讀取速度)

In [ ]:
import shutil
from huggingface_hub import snapshot_download

print("【階段 1】檢查雲端硬碟模型...")
if not os.path.exists(DRIVE_OBSERVER_PATH):
    print("⚠️ 雲端硬碟未發現 Qwen2-7B-Instruct，開始從 HuggingFace 下載至雲端硬碟 (這將花費幾分鐘)...")
    # 只下載必要的模型檔，排除佔空間的 pt/bin (因為通常有 safetensors)
    snapshot_download(
        repo_id="Qwen/Qwen2-7B-Instruct", 
        local_dir=DRIVE_OBSERVER_PATH,
        ignore_patterns=["*.pt", "*.bin"] 
    )
    print("✅ 雲端硬碟下載完成！")
else:
    print("✅ 雲端硬碟已存在 Qwen2-7B-Instruct。")

print("\n【階段 2】建立 Colab 本地快取...")
if not os.path.exists(LOCAL_OBSERVER_PATH):
    print("🔄 開始將模型從雲端硬碟複製到 Colab 本地暫存空間...")
    print("   (此步驟可避免後續大量網路 I/O，依硬碟速度約需 1~3 分鐘，請耐心等候)")
    shutil.copytree(DRIVE_OBSERVER_PATH, LOCAL_OBSERVER_PATH)
    print("✅ 本地快取複製完成！")
else:
    print("✅ Colab 本地空間已有快取，準備就緒。")

## [Cell 5] Code: 環境與量化設定

In [ ]:
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("裝置：", DEVICE)

if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    major, _ = torch.cuda.get_device_capability()
    COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    COMPUTE_DTYPE = torch.float16
    print("⚠️ 沒抓到 GPU。請確認執行階段為 T4 GPU。")

# 統一使用 4-bit 量化，確保兩顆 7B 模型能塞入 T4 的 15GB VRAM
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
print("BNB 量化設定就緒")

## [Cell 6] Code: 載入雙模型

In [ ]:
# 1. 從「本地快取」載入審核者模型 (速度會比直接讀 Google Drive 快很多)
print("1. 載入審核者模型 (Qwen2-7B-Instruct)...")
reviewer_tokenizer = AutoTokenizer.from_pretrained(LOCAL_OBSERVER_PATH)
reviewer_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_OBSERVER_PATH,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    low_cpu_mem_usage=True # 保護系統 RAM
)
reviewer_model.eval()

# 2. 載入東坡扮演者模型
print("2. 載入東坡扮演者模型 (Base + LoRA)...")
dongpo_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
dongpo_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True # 保護系統 RAM
)
dongpo_model = PeftModel.from_pretrained(dongpo_base, LORA_WEIGHT_PATH)
dongpo_model.eval()

print("\n✅ 雙模型載入完畢！")
if torch.cuda.is_available():
    print(f"顯存佔用：{torch.cuda.memory_allocated()/1024**3:.2f} GB / 15.0 GB")

## [Cell 7] Code: 定義審核者 (Reviewer) 邏輯

In [ ]:
REVIEWER_SYSTEM = """你是一個歷史時間線審核系統。
你的任務是判斷使用者的問題中，是否包含宋代文人蘇軾（西元1037-1101年）不可能知道的現代人物、現代科技或後世事件（例如：賴清德、手機、網路、明朝等）。

請嚴格以 JSON 格式輸出，不要包含任何其他文字或 markdown 標籤：
- 若無時間衝突，輸出：{"has_conflict": false, "action": "pass"}
- 若有時間衝突，輸出：{"has_conflict": true, "action": "請回答『[現代事物名稱]是什麼？我不知道』"}"""

@torch.inference_mode()
def run_reviewer(user_question: str) -> dict:
    messages = [
        {"role": "system", "content": REVIEWER_SYSTEM},
        {"role": "user", "content": f"使用者問題：{user_question}\n請輸出 JSON："}
    ]
    text = reviewer_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = reviewer_tokenizer([text], return_tensors="pt").to(DEVICE)
    
    out = reviewer_model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1, # 低溫確保邏輯穩定
        do_sample=False,
        pad_token_id=reviewer_tokenizer.eos_token_id # 避免 Warning
    )
    raw_output = reviewer_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    # 解析 JSON (防呆處理，去除可能生成的 markdown 區塊)
    try:
        json_str = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_str:
            return json.loads(json_str.group())
        else:
            return json.loads(raw_output)
    except Exception as e:
        print(f"[審核者解析錯誤] 原文: {raw_output}")
        return {"has_conflict": False, "action": "pass"}

## [Cell 8] Code: 定義東坡扮演者邏輯與主流程

In [ ]:
SUDONGPO_SYSTEM = (
    "你現在是宋代文人蘇軾（蘇東坡）。"
    "一律使用繁體中文，融合文言與白話，展現豁達幽默的個性。"
)

@torch.inference_mode()
def run_dongpo(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SUDONGPO_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    text = dongpo_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = dongpo_tokenizer([text], return_tensors="pt").to(DEVICE)
    
    out = dongpo_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.8,
        do_sample=True,
        pad_token_id=dongpo_tokenizer.pad_token_id or dongpo_tokenizer.eos_token_id,
    )
    return dongpo_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def chat_pipeline(user_question: str):
    # 1. 審核者判定
    review_result = run_reviewer(user_question)
    has_conflict = review_result.get("has_conflict", False)
    action = review_result.get("action", "pass")
    
    # 2. 構建動態 Prompt
    if has_conflict:
        # 強制阻斷指令
        final_prompt = (
            f"【系統強制指令】\n"
            f"使用者詢問了你（蘇軾）時代不存在的事物。請執行以下動作，將其融入你的回答中：\n{action}\n\n"
            f"【使用者問題】\n{user_question}"
        )
    else:
        # 正常對話
        final_prompt = f"【使用者問題】\n{user_question}"
        
    # 3. 東坡生成回應
    dongpo_response = run_dongpo(final_prompt)
    
    # 顯示過程與結果
    print("=" * 60)
    print(f"【審核者判定】衝突: {has_conflict} | 動作: {action}")
    print("-" * 60)
    print("【蘇東坡回答】")
    print(dongpo_response)
    print("=" * 60)
    
    # 釋放 VRAM 確保迴圈執行穩定
    torch.cuda.empty_cache()
    
    return dongpo_response

print("協同管線就緒！")

## [Cell 9] Code: 清德測試

In [ ]:
question = "請問你知道賴清德嗎？"
_ = chat_pipeline(question)

[Cell 10] Code: 互動測試

In [ ]:
print("輸入問題開始對話，輸入 'exit' 或 'quit' 結束。\n")
while True:
    try:
        user_input = input("您的問題：").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n[系統] 已中止。"); break
    if not user_input:
        continue
    if user_input.lower() in ("exit", "quit"):
        print("[系統] 結束。"); break
    
    chat_pipeline(user_input)
    print()